# Lesson 6: The Commission Report - Joining DataFrames

The agents are knocking on your door. They want their commission reports.

The problem? Property details are in `property_listings.csv` and commission payments are in `agent_commissions.json`. They live in separate files, like two halves of a contract that need to be matched.

**Joins** are how you match records across DataFrames - the same way a real estate transaction matches a buyer with a property, an agent, and a price.

### The Four Join Types - A Real Estate Metaphor

| Join Type | Real Estate Analogy |
|-----------|--------------------|
| **Inner Join** | Only close deals - properties that sold AND have commission records |
| **Left Join** | All listings on the market, with commission data attached where available |
| **Right Join** | All commission records, with property details attached where available |
| **Full Outer Join** | Everything - matched deals, unsold listings, and orphaned commission records |
| **Anti Join** | Properties that never sold - the unsold inventory report |
| **Broadcast Join** | Speed trick - send a small table to every worker node to avoid shuffling |

In this lesson you will build the commission report step by step, using each join type along the way.


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, broadcast, round, avg, count

spark = SparkSession.builder \
    .appName("Lesson6_CommissionReport") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# Load property listings
listings = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("data/property_listings.csv")

# Load agent commissions (JSON array format)
commissions = spark.read \
    .option("multiline", "true") \
    .json("data/agent_commissions.json")

print(f"Listings rows   : {listings.count()}")
print(f"Commissions rows: {commissions.count()}")

print("\nListings schema:")
listings.printSchema()

print("Commissions schema:")
commissions.printSchema()

Listings rows   : 510


Commissions rows: 363

Listings schema:
root
 |-- property_id: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- zip_code: integer (nullable = true)
 |-- property_type: string (nullable = true)
 |-- bedrooms: integer (nullable = true)
 |-- bathrooms: double (nullable = true)
 |-- sqft: double (nullable = true)
 |-- list_price: double (nullable = true)
 |-- year_built: double (nullable = true)
 |-- neighborhood: string (nullable = true)
 |-- status: string (nullable = true)
 |-- listing_date: date (nullable = true)
 |-- agent_id: string (nullable = true)

Commissions schema:
root
 |-- agent_id: string (nullable = true)
 |-- commission_amount: double (nullable = true)
 |-- commission_rate: double (nullable = true)
 |-- property_id: string (nullable = true)
 |-- sale_date: string (nullable = true)
 |-- sale_price: long (nullable = true)



## Inner Join - The Perfect Match

An inner join returns **only the rows that match on both sides**.

In our dataset: a property appears in the result only if it has a commission record.  
A commission record appears only if its `property_id` exists in the listings.

This is the classic "closed deals" report - every row represents a property that was both listed and sold.

```python
df_left.join(df_right, on="join_key", how="inner")
```


In [2]:
# Inner join: only properties that have a matching commission record
closed_deals = listings.join(commissions, on="property_id", how="inner")

print(f"Listings count     : {listings.count()}")
print(f"Commissions count  : {commissions.count()}")
print(f"Inner join result  : {closed_deals.count()} rows")
print("(Only properties that appear in BOTH DataFrames are included)")

print("\n--- Sample Closed Deals ---")
closed_deals.select(
    "property_id", "address", "neighborhood", "property_type",
    "list_price", "sale_price", "commission_amount", "sale_date"
).show(5)

Listings count     : 510


Commissions count  : 363


Inner join result  : 371 rows
(Only properties that appear in BOTH DataFrames are included)

--- Sample Closed Deals ---


+-----------+------------+------------+-------------+----------+----------+-----------------+----------+
|property_id|     address|neighborhood|property_type|list_price|sale_price|commission_amount| sale_date|
+-----------+------------+------------+-------------+----------+----------+-----------------+----------+
|  PROP-0405|225 River Rd|    Parkview|         NULL|  252780.0|    214633|          4292.66|2026-01-24|
|  PROP-0438|3844 Oak Ave|    Downtown|    Apartment|  694061.0|    323425|          9702.75|2025-12-04|
|  PROP-0035| 261 Oak Ave|    Hillside|        House|  415308.0|    495182|          9903.64|2026-02-26|
|  PROP-0234| 925 Hill Dr|    Parkview|        House|  508766.0|    667002|         20010.06|2026-01-03|
|  PROP-0086|7654 Main St|     Seaside|    Townhouse| 1003062.0|    401466|         10036.65|2025-09-24|
+-----------+------------+------------+-------------+----------+----------+-----------------+----------+
only showing top 5 rows


## Left Join - All Properties, Some With Commissions

A left join keeps **every row from the left DataFrame** and attaches matching data from the right.  
Where there is no match, the right-side columns are filled with `null`.

In real estate terms: show me every listing on our books. If it sold, show the commission details too.  
If it hasn't sold yet, show the listing anyway with nulls for the commission columns.

This is the most common join in practice - you rarely want to silently drop rows.


In [3]:
# Left join: all listings, commission data where available
all_listings_with_commissions = listings.join(
    commissions, on="property_id", how="left"
)

print(f"Left join result: {all_listings_with_commissions.count()} rows")
print("(All listings retained; commission columns are null for unsold properties)")

# Show some rows where commission_amount is null (unsold properties)
print("\n--- Unsold Properties (null commission_amount) ---")
all_listings_with_commissions.filter(
    col("commission_amount").isNull()
).select(
    "property_id", "address", "neighborhood", "status", "commission_amount"
).show(5)

# Count how many have commissions vs not
sold_count = all_listings_with_commissions.filter(col("commission_amount").isNotNull()).count()
unsold_count = all_listings_with_commissions.filter(col("commission_amount").isNull()).count()
print(f"\nWith commission (sold)   : {sold_count}")
print(f"Without commission (unsold): {unsold_count}")

Left join result: 610 rows
(All listings retained; commission columns are null for unsold properties)

--- Unsold Properties (null commission_amount) ---


+-----------+-------------+------------+--------+-----------------+
|property_id|      address|neighborhood|  status|commission_amount|
+-----------+-------------+------------+--------+-----------------+
|  PROP-0001|7370 River Rd|   Riverside|    Sold|             NULL|
|  PROP-0003| 5291 Hill Dr|    Hillside| Pending|             NULL|
|  PROP-0006| 5678 Hill Dr|     Seaside|For Sale|             NULL|
|  PROP-0007| 8422 Hill Dr|  Greenfield|    Sold|             NULL|
|  PROP-0009| 7049 Oak Ave|    Parkview|    Sold|             NULL|
+-----------+-------------+------------+--------+-----------------+
only showing top 5 rows



With commission (sold)   : 371
Without commission (unsold): 239


## Right Join and Full Outer Join

**Right join** is the mirror of left join - keep all rows from the **right** DataFrame, attach left-side data where it matches.

**Full outer join** keeps **all rows from both sides**. Where there's no match, nulls fill in on the missing side.

In practice:
- Right join: "show me every commission record, with property details if available" (catches orphaned commissions)
- Full outer join: "show me everything - I want to see what's matched and what's dangling on either side"


In [4]:
# Right join: all commission records, property details where available
right_join_result = listings.join(commissions, on="property_id", how="right")
print(f"Right join result: {right_join_result.count()} rows")
print("(All commission records kept; property columns null if property_id not in listings)")

# Check for orphaned commission records (property not in listings)
orphaned_commissions = right_join_result.filter(col("address").isNull())
print(f"Orphaned commission records (no matching listing): {orphaned_commissions.count()}")

# Full outer join: everything from both sides
full_outer_result = listings.join(commissions, on="property_id", how="outer")
print(f"\nFull outer join result: {full_outer_result.count()} rows")
print("(Every row from both DataFrames; nulls where match is missing on either side)")

# Summary comparison
print("\n--- Join Type Row Count Comparison ---")
print(f"  Inner join : {closed_deals.count()} rows (only matched)")
print(f"  Left join  : {all_listings_with_commissions.count()} rows (all listings)")
print(f"  Right join : {right_join_result.count()} rows (all commissions)")
print(f"  Outer join : {full_outer_result.count()} rows (everything)")

Right join result: 371 rows
(All commission records kept; property columns null if property_id not in listings)


Orphaned commission records (no matching listing): 0



Full outer join result: 610 rows
(Every row from both DataFrames; nulls where match is missing on either side)

--- Join Type Row Count Comparison ---


  Inner join : 371 rows (only matched)


  Left join  : 610 rows (all listings)


  Right join : 371 rows (all commissions)


  Outer join : 610 rows (everything)


## Handling Column Name Conflicts

When both DataFrames have a column with the **same name** (other than the join key),  
PySpark keeps both columns in the result - but you can no longer reference them by name alone.

For example, if `listings` has an `agent_id` column AND `commissions` also has `agent_id`,  
after a join you'll have two `agent_id` columns. PySpark won't know which one you mean.

**Solutions:**
1. Alias the DataFrames before joining and qualify column references
2. Rename conflicting columns before the join
3. Select only the columns you need after the join


In [5]:
# Demonstrate the duplicate column problem and the fix

# Both DataFrames have 'agent_id' - after join we have two of them
joined_raw = listings.join(commissions, on="property_id", how="inner")
print("Columns after inner join (notice duplicate agent_id):")
print([c for c in joined_raw.columns if c in ["agent_id", "property_id"]])
print("All columns:", joined_raw.columns)

# FIX 1: Rename before joining
commissions_renamed = commissions.withColumnRenamed("agent_id", "comm_agent_id")
clean_join = listings.join(commissions_renamed, on="property_id", how="inner")
print("\nAfter renaming agent_id in commissions:")
print([c for c in clean_join.columns if "agent" in c])

# FIX 2: Select only what you need after the join (most common approach)
print("\n--- Clean Commission Report (selected columns only) ---")
commission_report = listings.join(commissions, on="property_id", how="inner").select(
    "property_id",
    listings["agent_id"].alias("agent_id"),
    "address",
    "neighborhood",
    "property_type",
    "list_price",
    "sale_price",
    "commission_rate",
    "commission_amount",
    "sale_date"
)
commission_report.show(5)

Columns after inner join (notice duplicate agent_id):


['property_id', 'agent_id', 'agent_id']
All columns: ['property_id', 'address', 'city', 'zip_code', 'property_type', 'bedrooms', 'bathrooms', 'sqft', 'list_price', 'year_built', 'neighborhood', 'status', 'listing_date', 'agent_id', 'agent_id', 'commission_amount', 'commission_rate', 'sale_date', 'sale_price']

After renaming agent_id in commissions:
['agent_id', 'comm_agent_id']

--- Clean Commission Report (selected columns only) ---


+-----------+--------+------------+------------+-------------+----------+----------+---------------+-----------------+----------+
|property_id|agent_id|     address|neighborhood|property_type|list_price|sale_price|commission_rate|commission_amount| sale_date|
+-----------+--------+------------+------------+-------------+----------+----------+---------------+-----------------+----------+
|  PROP-0405| AGT-012|225 River Rd|    Parkview|         NULL|  252780.0|    214633|           0.02|          4292.66|2026-01-24|
|  PROP-0438| AGT-016|3844 Oak Ave|    Downtown|    Apartment|  694061.0|    323425|           0.03|          9702.75|2025-12-04|
|  PROP-0035| AGT-019| 261 Oak Ave|    Hillside|        House|  415308.0|    495182|           0.02|          9903.64|2026-02-26|
|  PROP-0234| AGT-004| 925 Hill Dr|    Parkview|        House|  508766.0|    667002|           0.03|         20010.06|2026-01-03|
|  PROP-0086| AGT-001|7654 Main St|     Seaside|    Townhouse| 1003062.0|    401466|      

## Joining with Neighborhood Stats

The investment team wants to **enrich** the listings with neighborhood-level context:  
school ratings, crime rates, median income. This data lives in `neighborhood_stats.parquet`.

This is a classic enrichment join - the property rows gain additional context columns  
from a lookup table keyed on `neighborhood`.


In [6]:
# Load neighborhood stats
neighborhood_stats = spark.read.parquet("data/neighborhood_stats.parquet")

print(f"Neighborhood stats rows: {neighborhood_stats.count()}")
neighborhood_stats.printSchema()
neighborhood_stats.show()

# Drop null neighborhoods from listings before join
listings_clean = listings.dropna(subset=["neighborhood"])

# Enrich listings with neighborhood context
enriched_listings = listings_clean.join(
    neighborhood_stats, on="neighborhood", how="left"
)

print(f"\nEnriched listings rows: {enriched_listings.count()}")
print("\n--- Listings Enriched with Neighborhood Stats ---")
enriched_listings.select(
    "property_id", "address", "neighborhood", "list_price",
    "school_rating", "crime_rate_per_1000", "median_income"
).show(8)

Neighborhood stats rows: 8
root
 |-- neighborhood: string (nullable = true)
 |-- median_income: long (nullable = true)
 |-- school_rating: double (nullable = true)
 |-- crime_rate_per_1000: double (nullable = true)
 |-- population: long (nullable = true)
 |-- avg_commute_time: long (nullable = true)
 |-- parks_count: long (nullable = true)



+------------+-------------+-------------+-------------------+----------+----------------+-----------+
|neighborhood|median_income|school_rating|crime_rate_per_1000|population|avg_commute_time|parks_count|
+------------+-------------+-------------+-------------------+----------+----------------+-----------+
|    Downtown|       145967|          7.6|              16.51|     23907|              23|          4|
|   Riverside|       138989|          8.3|              24.01|     11159|              16|          9|
|    Hillside|        85079|          9.8|              22.18|     34158|              31|          8|
|     Oakwood|       131556|          8.1|               9.57|     25293|              41|          3|
|    Parkview|        99323|          9.3|                9.4|     38932|              43|         13|
|     Seaside|        95320|          8.2|               9.02|     38387|              19|          8|
|     Midtown|       142098|          6.2|              23.45|     45750|


Enriched listings rows: 487

--- Listings Enriched with Neighborhood Stats ---


+-----------+-------------+------------+----------+-------------+-------------------+-------------+
|property_id|      address|neighborhood|list_price|school_rating|crime_rate_per_1000|median_income|
+-----------+-------------+------------+----------+-------------+-------------------+-------------+
|  PROP-0001|7370 River Rd|   Riverside| 1268597.0|          8.3|              24.01|       138989|
|  PROP-0002|960 Park Blvd|     Oakwood|  835503.0|          8.1|               9.57|       131556|
|  PROP-0003| 5291 Hill Dr|    Hillside|  487283.0|          9.8|              22.18|        85079|
|  PROP-0004| 5834 Oak Ave|    Downtown|  733117.0|          7.6|              16.51|       145967|
|  PROP-0005|566 Park Blvd|    Downtown|  951151.0|          7.6|              16.51|       145967|
|  PROP-0006| 5678 Hill Dr|     Seaside|  550782.0|          8.2|               9.02|        95320|
|  PROP-0007| 8422 Hill Dr|  Greenfield|  174042.0|          7.3|              22.05|       139909|


## Anti-Join - Finding the Unmatched

An anti-join is the opposite of an inner join. It returns rows from the **left DataFrame  
that have NO match in the right DataFrame**.

In real estate: give me every listing that has **never generated a commission record**.  
That's the unsold inventory report - properties sitting on the market with no buyer.

```python
df_left.join(df_right, on="key", how="left_anti")
```


In [7]:
# Left anti-join: listings with NO commission record = unsold inventory
unsold_inventory = listings.join(commissions, on="property_id", how="left_anti")

print(f"Total listings       : {listings.count()}")
print(f"Unsold (no commission): {unsold_inventory.count()}")

print("\n--- Unsold Inventory Report ---")
unsold_inventory.select(
    "property_id", "address", "neighborhood", "property_type",
    "list_price", "status", "listing_date"
).orderBy("listing_date").show(10)

# How does unsold inventory break down by neighborhood?
print("--- Unsold Inventory by Neighborhood ---")
unsold_inventory.groupBy("neighborhood") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

Total listings       : 510


Unsold (no commission): 239

--- Unsold Inventory Report ---


+-----------+------------+------------+-------------+----------+--------+------------+
|property_id|     address|neighborhood|property_type|list_price|  status|listing_date|
+-----------+------------+------------+-------------+----------+--------+------------+
|  PROP-0148|2305 Hill Dr|    Parkview|    Apartment|  171067.0|For Sale|        NULL|
|  PROP-0271|1642 Hill Dr|     Seaside|         NULL|  723881.0| Pending|        NULL|
|  PROP-0155|1080 Main St|    Parkview|        Condo|  681740.0|    Sold|        NULL|
|  PROP-0038|6963 Hill Dr|    Downtown|        Villa| 1076921.0|For Sale|        NULL|
|  PROP-0003|5291 Hill Dr|    Hillside|        House|  487283.0| Pending|        NULL|
|  PROP-0264|5405 Oak Ave|    Downtown|    Apartment|  645919.0|    NULL|        NULL|
|  PROP-0182|104 River Rd|   Riverside|        House|  826315.0|For Sale|        NULL|
|  PROP-0071|9889 Main St|   Riverside|        House| 1137626.0|For Sale|        NULL|
|  PROP-0193|8408 Main St|    Parkview|    

+------------+-----+
|neighborhood|count|
+------------+-----+
|     Seaside|   37|
|   Riverside|   31|
|  Greenfield|   30|
|    Hillside|   29|
|    Parkview|   27|
|    Downtown|   27|
|     Midtown|   24|
|     Oakwood|   24|
|        NULL|   10|
+------------+-----+



In [8]:
# Broadcast join: use when one table is small enough to fit in memory on every worker
# neighborhood_stats has only 8 rows - perfect broadcast candidate

print("--- Broadcast Join with neighborhood_stats (8 rows) ---")
print("The broadcast() hint tells Spark to send this small table to every executor")
print("instead of doing an expensive shuffle of the large listings DataFrame.\n")

# Without broadcast hint (Spark may still broadcast automatically for small tables)
# With explicit broadcast hint:
enriched_broadcast = listings_clean.join(
    broadcast(neighborhood_stats), on="neighborhood", how="left"
)

print(f"Broadcast join result rows: {enriched_broadcast.count()}")

# You can also set the auto-broadcast threshold
print("\nCurrent auto-broadcast threshold:")
print(spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

# Increase to 20MB for small-table optimization
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 20 * 1024 * 1024)
print("Set to 20MB:", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

print("\n--- Sample Broadcast Join Result ---")
enriched_broadcast.select(
    "neighborhood", "property_type", "list_price",
    "school_rating", "median_income", "avg_commute_time"
).show(6)

--- Broadcast Join with neighborhood_stats (8 rows) ---
The broadcast() hint tells Spark to send this small table to every executor
instead of doing an expensive shuffle of the large listings DataFrame.



Broadcast join result rows: 487

Current auto-broadcast threshold:
10485760b
Set to 20MB: 20971520

--- Sample Broadcast Join Result ---


+------------+-------------+----------+-------------+-------------+----------------+
|neighborhood|property_type|list_price|school_rating|median_income|avg_commute_time|
+------------+-------------+----------+-------------+-------------+----------------+
|   Riverside|        House| 1268597.0|          8.3|       138989|              16|
|     Oakwood|        House|  835503.0|          8.1|       131556|              41|
|    Hillside|        House|  487283.0|          9.8|        85079|              31|
|    Downtown|        Condo|  733117.0|          7.6|       145967|              23|
|    Downtown|        Villa|  951151.0|          7.6|       145967|              23|
|     Seaside|        House|  550782.0|          8.2|        95320|              19|
+------------+-------------+----------+-------------+-------------+----------------+
only showing top 6 rows


## Lesson 6 Wrap-Up

You can now combine any number of DataFrames to build the commission report the agents were waiting for.

**What you learned:**
- **Inner join** (`how="inner"`) - only matched rows from both sides
- **Left join** (`how="left"`) - all left rows, right data where available
- **Right join** (`how="right"`) - all right rows, left data where available
- **Full outer join** (`how="outer"`) - all rows from both, nulls for non-matches
- **Anti-join** (`how="left_anti"`) - left rows with NO match on the right
- **Broadcast join** (`broadcast()` hint) - performance optimization for small lookup tables
- Handling duplicate column names with aliases and column selection

**Key rule of thumb:**  
Prefer `left` joins over `inner` joins unless you are certain every row on the left has a match.  
Silent row drops from inner joins are one of the most common sources of incorrect aggregations.

---

**Next: Lesson 7 - Market Pulse (Window Functions!)**  
Who are the top 3 agents in each neighborhood? What's the running total of sales?  
Window functions let you rank, compare, and analyze trends **without collapsing your rows**.
